In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split

/home/daan/OHL-AI-Week/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
GK_DATA = Path("GK_Data")
def load_keeper_match_kpis(player_id, match_dirs_str):
    """Load player_kpis for a keeper from all his matches."""
    all_kpis = []
    if not isinstance(match_dirs_str, str) or not match_dirs_str:
        return all_kpis

    for match_dir in match_dirs_str.split("|"):
        match_dir = match_dir.strip()
        if not match_dir:
            continue
        for cs_dir in (GK_DATA / "competitions").iterdir():
            if not cs_dir.is_dir():
                continue
            pkpi_path = cs_dir / match_dir / "player_kpis.json"
            if pkpi_path.exists():
                with open(pkpi_path) as f:
                    data = json.load(f).get("data", {})
                for side in ["squadHome", "squadAway"]:
                    for player in data.get(side, {}).get("players", []):
                        if (
                            player["id"] == player_id
                            and player.get("position") == "GOALKEEPER"
                            and player.get("matchShare", 0) >= 0.5
                        ):
                            kpis = {k["kpiId"]: k["value"] for k in player.get("kpis", [])}
                            kpis["matchShare"] = player["matchShare"]
                            kpis["playDuration"] = player.get("playDuration", 0)
                            all_kpis.append(kpis)
                break  
    return all_kpis

dataset = pd.read_csv(GK_DATA / "gk_dataset_final.csv")

with open(GK_DATA / "player_kpi_definitions.json") as f:
    kpi_defs = {d["id"]: d["name"] for d in json.load(f).get("data", [])}

all_match_rows = []

for _, row in dataset.iterrows():
    match_kpis = load_keeper_match_kpis(row["playerId"], row["origin_match_dirs"])
    
    if match_kpis:
        for match_data in match_kpis:
            entry = row.drop(['origin_match_dirs', 'current_match_dirs']).to_dict()
            
            for kpi_id, value in match_data.items():
                kpi_name = kpi_defs.get(kpi_id, f"KPI_{kpi_id}")
                entry[kpi_name] = value
            
            all_match_rows.append(entry)

df = pd.DataFrame(all_match_rows)
print(f"Done! Created a dataset with {len(df)} total match-performance rows.")
df.sample(5)

Done! Created a dataset with 26127 total match-performance rows.


,playerId,name,age,birthdate,status,origin_team,origin_comp,origin_season,origin_median,origin_matches,...,LOST_GROUND_DUELS_IN_PITCH_POSITION_OPPONENT_BOX,ASSISTS_BY_ACTION_SAVE,ASSISTS_AT_PHASE_SECOND_BALL,BYPASSED_OPPONENTS_FROM_PACKING_ZONE_WR,BYPASSED_OPPONENTS_NUMBER_FROM_PACKING_ZONE_WR,BALL_LOSS_REMOVED_TEAMMATES_FROM_PACKING_ZONE_WL,BYPASSED_DEFENDERS_RECEIVING_TO_PITCH_POSITION_FIRST_THIRD,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_LANE_RIGHT_WING,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_OPPONENT_BOX,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_IN_PITCH_POSITION_OPPONENT_BOX
19188,1798,Brice Samba,31.0,1994-04-25,STAYED,RC Lens,ligue1,2023-2024,0.771221,88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8910,136441,Dmytro Ledviy,22.0,2003-08-26,STAYED,Rukh Lviv,premier_liga_ukraine,2023-2024,0.401355,57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25983,78774,Marian Aioani,26.0,1999-11-07,STAYED,FC Rapid Bukarest,superliga_romania,2024-2025,0.456571,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7408,54883,Marvin Keller,23.0,2002-07-03,STAYED,FC Winterthur,super_league_switzerland,2023-2024,0.664379,79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8861,136441,Dmytro Ledviy,22.0,2003-08-26,STAYED,Rukh Lviv,premier_liga_ukraine,2023-2024,0.401355,57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.loc[df['direction'] == 'DOWN', 'step'] = -df['step'].abs()
print(df.groupby('direction')['step'].mean())

direction
DOWN   -0.176983
NONE    0.000000
UP      0.169567
Name: step, dtype: float64


In [9]:
import pandas as pd

# 1. Define your essential identifiers
# IMPORTANT: Include 'current_matches' so Match 1 is distinct from Match 2
metadata = [
    'playerId', 'name', 'age', 'status', 'current_matches',
    'current_team', 'current_comp', 'step'
]

# 2. Calculate Correlations
numeric_df = df.select_dtypes(include=['number'])
# Using the absolute value to catch both strong positive and negative drivers of 'step'
correlations = numeric_df.corr()['step'].abs().sort_values(ascending=False)

# 3. Filter KPIs (Correlation > 0.1)
significant_kpis = correlations[correlations > 0.1].index.tolist()

# 4. Combine and deduplicate the column names list
final_column_names = list(set(metadata + significant_kpis))

# 5. Create the useful dataframe
useful_df = df[final_column_names].copy()

# 6. HANDLE DUPLICATES
# This removes rows where EVERY column is identical
useful_df = useful_df.drop_duplicates()

# 7. SORT CHRONOLOGICALLY
# Now we order it by player and match sequence
useful_df['current_matches'] = pd.to_numeric(useful_df['current_matches'], errors='coerce')
useful_df = useful_df.sort_values(by=['playerId', 'current_matches'])

useful_df.to_csv('useful_keeper_dataset2.csv', index=False)

print(f"Final Dataset Shape: {useful_df.shape}")
print(f"Top 5 Drivers of Rank Movement ('step'):\n{correlations.head(6)}")

Final Dataset Shape: (3386, 111)
Top 5 Drivers of Rank Movement ('step'):
BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_OWN_BOX        1.0
step                                                                    1.0
BALL_LOSS_ADDED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_MIDDLE_THIRD    1.0
BYPASSED_DEFENDERS_RECEIVING_AT_PHASE_IN_POSSESSION                     1.0
BYPASSED_OPPONENTS_RECEIVING_FROM_PACKING_ZONE_CM                       1.0
BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_BY_ACTION_HEADER                   1.0
Name: step, dtype: float64


In [10]:
# Count total identical rows
exact_duplicates = useful_df.duplicated().sum()
print(f"Total exact duplicate rows: {exact_duplicates}")

# To see them (optional)
if exact_duplicates > 0:
    print(useful_df[useful_df.duplicated(keep=False)])

Total exact duplicate rows: 0
